In [35]:
# IMPORTS
import numpy as np
import pandas as pd
import requests

#Fin Data Sources
import yfinance as yf
import pandas_datareader as pdr

#Data viz
import plotly.graph_objs as go
import plotly.express as px

import time
from datetime import date

# for graphs
import matplotlib.pyplot as plt

# ignore warnings
import warnings
warnings.filterwarnings('ignore')

### Question 1: IPO Filings Web Scraping and Data Processing

**What's the total sum ($m) of 2023 filings that happened on Fridays?**

Re-use the [Code Snippet 1] example to get the data from web for this endpoint: https://stockanalysis.com/ipos/filings/
Convert the 'Filing Date' to datetime(), 'Shares Offered' to float64 (if '-' is encountered, populate with NaNs).
Define a new field 'Avg_price' based on the "Price Range", which equals to NaN if no price is specified, to the price (if only one number is provided), or to the average of 2 prices (if a range is given).
You may be inspired by the function `extract_numbers()` in [Code Snippet 4], or you can write your own function to "parse" a string.
Define a column "Shares_offered_value", which equals to "Shares Offered" * "Avg_price" (when both columns are defined; otherwise, it's NaN)

Find the total sum in $m (millions of USD, closest INTEGER number) for all filings during 2023, which happened on Fridays (`Date.dt.dayofweek()==4`). You should see 32 records in total, 25 of it is not null.

(additional: you can read about [S-1 IPO filing](https://www.dfinsolutions.com/knowledge-hub/thought-leadership/knowledge-resources/what-s-1-ipo-filing) to understand the context)

In [36]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3',
}

url = "https://stockanalysis.com/ipos/filings/"
response = requests.get(url, headers=headers)

ipo_dfs = pd.read_html(response.text)

ipos_2023 = ipo_dfs[0]

ipos_2023.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 325 entries, 0 to 324
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Filing Date     325 non-null    object
 1   Symbol          325 non-null    object
 2   Company Name    325 non-null    object
 3   Price Range     325 non-null    object
 4   Shares Offered  325 non-null    object
dtypes: object(5)
memory usage: 12.8+ KB


In [37]:
ipos_2023.head()

,Filing Date,Symbol,Company Name,Price Range,Shares Offered
0,"May 3, 2024",TBN,Tamboran Resources Corporation,-,-
1,"Apr 29, 2024",HWEC,"HW Electro Co., Ltd.",$3.00,3750000
2,"Apr 29, 2024",DTSQ,DT Cloud Star Acquisition Corporation,$10.00,6000000
3,"Apr 26, 2024",EURK,Eureka Acquisition Corp,$10.00,5000000
4,"Apr 26, 2024",HDL,Super Hi International Holding Ltd.,-,-


In [38]:
# convert Fillings Date to datetime
ipos_2023['Filing Date'] = pd.to_datetime(ipos_2023['Filing Date'])

# choose only fillings for 2023
ipos_2023 = ipos_2023[ipos_2023['Filing Date'].dt.year == 2023]

# replace "-" with NaN in Shares Offered and to float64
ipos_2023['Shares Offered'] = ipos_2023['Shares Offered'].replace('-', np.nan).astype(float)

# Define a new field 'Avg_price' based on the "Price Range", which equals to NaN if no price is specified, to the price
# (if only one number is provided), or to the average of 2 prices (if a range is given).
#You may be inspired by the function `extract_numbers()` in [Code Snippet 4], or you can write your own function to
# "parse" a string.

# Function to parse price range string and calculate average price
def calculate_avg_price(price_range):
    # remove $ from price
    price_range = price_range.replace('$', '')

    if price_range == '-':
        return np.nan
    elif '-' in price_range:
        prices = price_range.split('-')
        return (float(prices[0]) + float(prices[1])) / 2
    else:
        return float(price_range)

# Define a new field 'Avg_price' based on the "Price Range"
ipos_2023['Avg_price'] = ipos_2023['Price Range'].apply(calculate_avg_price)

# Define a column "Shares_offered_value", which equals to "Shares Offered" * "Avg_price" (when both columns are defined; otherwise, it's NaN)
ipos_2023['Shares_offered_value'] = ipos_2023['Shares Offered'] * ipos_2023['Avg_price']

In [39]:
ipos_2023[ipos_2023['Filing Date'].dt.dayofweek == 4].shape

(32, 7)

In [42]:
# Find the total sum in $m (millions of USD, closest INTEGER number) for all filings during 2023, which happened on Fridays (`Date.dt.dayofweek()==4`).
# You should see 32 records in total, 25 of it is not null. The total sum should be 1,000,000,000 USD.
round(ipos_2023[ipos_2023['Filing Date'].dt.dayofweek == 4]['Shares_offered_value'].sum()/1e6, 0)

286.0

In [29]:
ipos_2023.head()

,Filing Date,Symbol,Company Name,Price Range,Shares Offered,Avg_price,Shares_offered_value
0,2024-05-03,TBN,Tamboran Resources Corporation,-,NaN,NaN,NaN
1,2024-04-29,HWEC,"HW Electro Co., Ltd.",$3.00,3750000.0,3.0,11250000.0
2,2024-04-29,DTSQ,DT Cloud Star Acquisition Corporation,$10.00,6000000.0,10.0,60000000.0
3,2024-04-26,EURK,Eureka Acquisition Corp,$10.00,5000000.0,10.0,50000000.0
4,2024-04-26,HDL,Super Hi International Holding Ltd.,-,NaN,NaN,NaN
